## Preproceso para la predicción mensual


In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
import collections
from IPython.display import display
import seaborn as sns
import re
from tqdm import tqdm_notebook as progress_bar
from difflib import SequenceMatcher
from itertools import combinations
import os

# sys.path.append('../../src')
# from comparar_distribuciones import *

# sys.path.append("../../../spike")
# import SpikePy as sp

import snowflake.connector as snow
import getpass as gp

from sqlalchemy import create_engine
plt.style.use('fivethirtyeight')
#Usar kernel spikelabs_env_3

/opt/miniconda3/envs/spikelabs_env_3/lib/python3.6/site-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


## Setear periodo 
Debiese ser el único input

In [2]:
#periodo a predecir 'mes(3letras)año'
hoy =  pd.to_datetime("today")
fecha_prediccion =  hoy - pd.DateOffset(months=1, day=1)
periodo_prediccion = fecha_prediccion.year*100 + fecha_prediccion.month

pre_ges = 0  # hubo cambio ges 5 meses antes del periodo de cierre 0 -> si no hubo cambio | 1 -> hubo cambio
hay_ges = 0  # hay /habrá cambio ges en los 5 meses siguientes 0 -> si no habrá camnbio | 1 -> habrá cambio


num_a_mes = {
    1: 'ene', 2: 'feb', 3: 'mar',  4: 'abr',  5: 'may',  6: 'jun',
    7: 'jul', 8: 'ago', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dic'
}
periodo = f"{num_a_mes[fecha_prediccion.month]}{fecha_prediccion:%y}"

In [3]:
snow_user = gp.getpass(prompt='Usuario')
snow_pass = gp.getpass(prompt = 'Password')

Usuario········
Password········


In [4]:
engine = create_engine(
    'snowflake://{user}:{password}@{account}/'.format(
        user=snow_user,
        password=snow_pass,
        account='isapre_colmena.us-east-1',
    )
)
try:
    connection = engine.connect()
    results = connection.execute('select current_version()').fetchone()
    print(results[0])
finally:
    connection.close()
    engine.dispose()

8.26.0


In [5]:
df_ges = pd.read_sql('SELECT * FROM EST.P_DDV_EST.JC_GES_PRED;', con=engine)
nombre_archivo = f'{hoy:%y.%m.%d}_GES_{fecha_prediccion:%Y%m}_{num_a_mes[fecha_prediccion.month]}{fecha_prediccion:%y}.gz'
ges_file = f'./input/preproceso/' + nombre_archivo
df_ges.to_csv(ges_file, index=False, compression='gzip')

In [6]:
#shift periodo
p = {'ene':'01','feb':'02','mar':'03','abr':'04','may':'05','jun':'06',
     'jul':'07','ago':'08','sep':'09','oct':'10','nov':'11','dic':'12'}

per = p[periodo[:3]]

t = str(int(per)%12+1)

if len(t) == 1:
    next_per = '0'+t
else:
    next_per = t        

if periodo[:3] == 'dic':
    agno = str(int(periodo[3:])+1)
else:
    agno = periodo[3:]

In [7]:
#todos los archivos necesarios
disco = '/mnt/disks/modelos/' #al bucket (original): /estudio/data/

inputs = 'input/preproceso/'

outputs = 'output/'

#inputs predicciones
in_pred = disco+'Proyecto_GES/Prediccion/input/prediccion/'

# print('Buscando los datos de ges del periodo...')

# temp_ges = []

# for root, subdirs, files in os.walk(inputs):
#      for filename in files:
#             if 'ges' in filename.lower():
#                 if periodo in filename:
#                     temp_ges.append(filename)
# try:
#     print('Los archivos encontrados son:...')

#     print(*temp_ges,sep='\n')
    
#     print(f'Se utilizara el archivo {temp_ges[0]} para el preproceso')
    
#     ges = temp_ges[0]
    
# except IndexError:
#     print('No hay archivos en el periodo, buscando en todos los periodos...')

#     for root, subdirs, files in os.walk('input'):
#          for filename in files:
#                 if 'ges' in filename.lower():
#                         temp_ges.append(filename)
#     print('Los archivos son:')
    
#     print(*temp_ges,sep='\n')
    
#     print('Modifique más abajo el archivo que desea utilizar')

ges = ges_file

In [8]:
print('Buscando los datos de ltv del periodo...')

temp_ltv = []

for root, subdirs, files in os.walk(inputs):
     for filename in files:
            if 'ltv' in filename.lower():
                if periodo in filename:
                    temp_ltv.append(filename)
try:
    print('Los archivos encontrados son:...')

    print(*temp_ltv,sep='\n')
    
    print(f'Se utilizara el archivo {temp_ltv[0]} para el preproceso')
    
    ltv = temp_ltv[0]
    
except IndexError:
    print('No hay archivos en el periodo, buscando en todos los periodos...')

    for root, subdirs, files in os.walk('input'):
         for filename in files:
                if 'ges' in filename.lower():
                        temp_ltv.append(filename)
    print('Los archivos son:')
    
    print(*temp_ltv,sep='\n')
    
    print('Modifique más abajo el archivo que desea utilizar')

Buscando los datos de ltv del periodo...
Los archivos encontrados son:...
24.06.01_LTV_202406_jun24.gz
Se utilizara el archivo 24.06.01_LTV_202406_jun24.gz para el preproceso


In [9]:
#datos auxiliares
div_reg = 'xlsx/Division Regiones.xlsx'
comunas = 'xlsx/cod comuna.xlsx'
pobreza = 'xlsx/nse_y_pobreza.xlsx'

In [10]:
#cargamos datos ltv
df_ltv =  pd.read_csv(inputs+ltv, sep=",", compression='gzip').rename(columns = lambda x: x.lower())
df_ltv['periodo'] = pd.to_datetime(df_ltv['periodo'], format='%Y%m')
# df_ltv.head()

In [11]:
df_ltv.head()

,rut_titular,id_titular,periodo,anno,mes,periodo_ult_vig,fechaingreso,antiguedad,sexo,num_cargas,...,margen_sum_future_2y,costos_avg_future_2y,ingresos_avg_future_2y,margen_avg_future_2y,costos_sum_future_3y,ingresos_sum_future_3y,margen_sum_future_3y,costos_avg_future_3y,ingresos_avg_future_3y,margen_avg_future_3y
0,018569093-8,89633011,2024-06-01,2024,6,202406,2020-07-06,46,F,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,019058979-K,92082441,2024-06-01,2024,6,202406,2018-10-31,67,F,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,014422778-6,68901436,2024-06-01,2024,6,202406,2012-08-24,141,M,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,015924389-3,76409491,2024-06-01,2024,6,202406,2004-03-12,242,M,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,019080744-4,92191266,2024-06-01,2024,6,202406,2023-06-23,11,M,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
#cargamos datos ges, solo algunas columnas
cols_to_use = ['id_titular','periodo','fld_pertermino','n_ges_mva_2m',
               'preexistencia_afi','preexistencia_car','gls_isapreant']

# df_ges = df pd.read_csv(inputs+ges, compression='zip') #usecols=cols_to_use,delimiter=';',

In [13]:
#generamos la variable huerfano y formateamos
df_ges['huerfano'] = ((df_ges['fld_pertermino'] < df_ges['periodo']) & (df_ges['fld_pertermino'] != 190001))
df_ges['periodo'] = pd.to_datetime(df_ges['periodo'], format='%Y%m')  #
# df_ges['periodo'] = (df_ges['periodo'] + pd.Timedelta('32 day')).apply(lambda f: f.replace(day=1))
df_ges = df_ges.rename(columns=lambda x: x.lower())
df_ges = df_ges.dropna(subset=['fld_pertermino'])
df_ges = df_ges.drop(['fld_pertermino'], axis=1)

In [14]:
df_ges.head()

,id_titular,id_ltv,periodo,cod_isapreant,gls_isapreant,fld_agecod,fld_agencianum,fld_termino,preexistencia_afi,n_ges_mva_2m,preexistencia_car,huerfano
0,75277331,75277331,2024-06-01,67,COLMENA GOLDEN CROSS S.A.,70,085-K,01/01/1900,0,0,0,False
1,77985791,77985791,2024-06-01,67,COLMENA GOLDEN CROSS S.A.,85,087-6,06/05/2023,0,0,0,True
2,67015121,67015121,2024-06-01,67,COLMENA GOLDEN CROSS S.A.,72,085-K,01/01/1900,1,0,0,False
3,27890351,27890351,2024-06-01,67,COLMENA GOLDEN CROSS S.A.,21,099-K,01/01/1900,0,1,1,False
4,38339396,38339396,2024-06-01,67,COLMENA GOLDEN CROSS S.A.,21,099-K,01/01/1900,1,0,1,False


In [15]:
#merge
df = pd.merge(df_ltv, df_ges, how='left', on=['periodo', 'id_titular'])

#### Shiftear periodos Fuga y VN

In [16]:
fugas = df.tipo_transac.isin(['D2', 'D3'])
index_fugas = fugas[fugas].index
periodos_fuga = df.loc[index_fugas, 'periodo']
df.loc[index_fugas, 'periodo'] = (periodos_fuga - pd.Timedelta('1 day')).apply(lambda f: f.replace(day=1))
df = df[df.renta_imponible >= 0]
df = df.drop(['anno', 'mes', 'fechaingreso'], axis=1)

### Datos preferente

In [17]:
def cat_linea_plan(x):
    for c, lst in categorias.items():
        if x in lst:
            return c
        
alto = ['CLINICA ALEMANA', 'CLINICA LAS CONDES', 'LOS ANDES', 'SAN CARLOS', 'ALEMANA-SC-LA-CSM / CLC', 
        'SAN CARLOS - LOS ANDES', 'SAN CARLOS - UC - LOS ANDES', 'BALTICO', 'SAN CARLOS - STA. MARIA - LOS ANDES',
        'ADRIATICO', 'SAN CARLOS-UC']
medio = ['SANTA MARIA', 'HOSP. CLINICO UNIVERSIDAD CATOLICA', 'CLINICA UNIVERSIDAD CATOLICA', 'TABANCURA', 
         'MEDITERRANEO', 'SAN CARLOS-UC', 'HOSP.CLINICO UNIVERSIDAD DE CHILE', 'INDISA INSTITUCIONAL', 'CLINICA INDISA']
bajo = ['DAVILA', 'BICENTENARIO-VESPUCIO-LAS LILAS',  
        'UC SJ 110 - BICENTENARIO - VESPUCIO', 'CORDILL.-BICENT.-AVANSALUD-SERVET', 'BICENTENARIO-TABANCURA-DAVILA', 
        'CITY COLMENA', 'BICENTENARIO-VESPUCIO', 'CLINICA CORDILLERA - CLINICA DAVILA', 
        'UC SJ - BICENTENARIO - VESPUCIO']
le = ['LIBRE ELECCION']

categorias = {'alto': alto, 'medio': medio, 'bajo': bajo, 'le': le}

In [18]:
#demora 1 min
cat = 'xlsx/PRM_Categoria.xlsx'
categoria = pd.read_excel(inputs+cat)
df = pd.merge(df, categoria[['cod_categoria', 'preferente']], how='left', 
                 left_on='categoria_cod', right_on='cod_categoria')
df['cat_linea_plan'] = df.preferente.apply(cat_linea_plan)

titulares = df.id_titular.unique()

In [19]:
replace_pref = {'UC ': 'UNIVERSIDAD CATOLICA',
                'CSM': 'STA. MARIA',
                'CLC': 'CLINICA LAS CONDES',
                'BICENT.': 'BICENTENARIO',
                'CORDILL.': 'CLINICA CORDILLERA'}

In [20]:
def limpiar_preferente(x):
    y = x
    for old, new in replace_pref.items():
        try:
            y = y.replace(old, new)
        except:
            pass
    return y

In [21]:
df.preferente = df.preferente.apply(limpiar_preferente)

### Datos Democráficos Extras

In [22]:
# demora 
division_reg = (pd.read_excel(inputs+div_reg, skiprows=1)
                [19::].rename(columns={'Unnamed: 3': 'norte_centro_sur'})
               [['COD_REGION', 'GLS_WEB_REGION', 'norte_centro_sur']])
division_reg['COD_REGION'] = division_reg.COD_REGION.astype('int')

cod_comuna = pd.read_excel(inputs+comunas)
nse_y_pobreza = pd.read_excel(inputs+pobreza)

nse_y_pobreza = pd.merge(nse_y_pobreza, cod_comuna, how='inner',
                        left_on='COMUNA', right_on='con_comuna_gls')
nse_y_pobreza['COD_REGION'] = nse_y_pobreza.cod_comuna.astype('str').str[:-3].astype('int')
full_comuna = pd.merge(nse_y_pobreza, division_reg, how='left',
                      on='COD_REGION')

#Asumiendo que tu base se llama df_res y tiene la variable `comuna_cod`
df = pd.merge(df, full_comuna, how='left',
                      left_on='comuna_cod', right_on='cod_comuna')

### Drop y limpieza 

In [23]:
df = df.rename(columns={'categoria_gls': 'categor__a_gls'})

In [24]:
drop = ['region_cod', 'comuna_cod', 'fecha_nacimiento', 'cod_sucursal', 'centrocostos_cod', 
        'detalle_producto', 'cod_categoria', 'con_comuna_gls', 'cod_comuna', 'COD_REGION',
        'GLS_WEB_REGION', 'comuna_gls']
df = df.drop(drop, axis=1)

In [25]:
cols_strip = ['region_gls', 'sucursal_gls', 'categor__a_gls', 'serie', 'tipo_plan', 'tipo_producto', 'actividad']
for col in cols_strip:
    df.loc[:, col] = df[col].str.rstrip()

## Generar Features

### Variable categórica de ges y pre-ges en ventana de tiempo
Setear en función de si habrá annuncio de alza por precio ges y (pre_ges)
y apertura de cartera por cambio (ges) en una ventana de tiempo (meses)

In [26]:
tamaño_ventana = 5 #meses a futuro
df[f'hay_preges_{tamaño_ventana}m'] = pre_ges
df[f'hay_ges_{tamaño_ventana}m'] = hay_ges

### Variable categórica de Fuga Voluntaria en ventana de tiempo
variable de entrenamient

In [27]:
df['fuga_5m'] = np.NaN

### Antigedad < 12 y GES = 1

In [28]:
df['ant_m12_y_ges'] = (df.antiguedad <= 12 - tamaño_ventana)

### Tuvo CIE complejo, tuvo gasto licencia

In [29]:
#propagar variables 
var = 'cie_complejo'
var_new = 'hubo_cie_complejo'
df_aux = df[['id_titular', 'periodo', var]]
df_aux = pd.pivot_table(df_aux, index='periodo', values=var, 
                               columns='id_titular', aggfunc=np.max)
df_aux = df_aux.sort_index().fillna(0)
df_aux = df_aux.rolling(window=100, min_periods=1).max()
df_aux = df_aux.stack().reset_index().rename(columns={0: var_new})
df = pd.merge(df, df_aux, how='left', on=['id_titular', 'periodo'])

In [30]:
df['hay_gasto_licencia'] = ((df.gasto_licencias != 0) | (df.gasto_licencias_excl != 0)) + 0

var = 'hay_gasto_licencia'
var_new = 'hubo_licencia'
df_aux = df[['id_titular', 'periodo', var]]
df_aux = pd.pivot_table(df_aux, index='periodo', values=var, 
                               columns='id_titular', aggfunc=np.max)
df_aux = df_aux.sort_index().fillna(0)
df_aux = df_aux.rolling(window=100, min_periods=1).max()
df_aux = df_aux.stack().reset_index().rename(columns={0: var_new})
datos = pd.merge(df, df_aux, how='left', on=['id_titular', 'periodo'])

In [31]:
f = f'ready_to_pred_{periodo}.csv'
#df.to_csv(outputs + f) dejo solamente la copia para la predicción, no tiene sentido estar duplicando este archivo
df.to_csv(in_pred + f) #apunta la carperta de inputs para la predicción

In [32]:
df.shape

(393336, 168)

In [33]:
print(df.n_ges_mva_2m.unique())
df.n_ges_mva_2m.head()

[ 0.  6.  4.  8.  1.  3.  2.  9. 13. 14.  5. 11.  7. 10. 12. nan 24. 22.
 23. 19. 15. 17. 18. 16. 25. 20. 33. 27. 32. 26. 21. 29. 35. 41. 37. 42.
 38. 45. 30. 46. 61. 43. 50. 28. 64.]


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: n_ges_mva_2m, dtype: float64